# How LLMs Work: A Beginner's Guide 🚀

In this notebook, we'll explore:
1. **How LLMs work** - The basic architecture
2. **Tokenization** - Breaking text into pieces LLMs understand
3. **Probability calculation** - How LLMs decide what word comes next
4. **Using HuggingFace models** - Practical examples

Let's think of an LLM like a **cricket batsman** 🏏:
- The batsman sees the ball (input text)
- Makes a decision based on pattern (what kind of shot?)
- Plays the shot (generates the next word)
- Repeats for the next ball

---

## Part 1: The Simple Pipeline - How LLMs Work

An LLM does 3 things:
1. **Read** the input text → Tokenize it
2. **Process** through layers → Learn patterns
3. **Output** → Predict the next word (with probability)

### Visual Flow:
```
Input: "Hello, how are"
   ↓
Tokenization: [Hello] [,] [how] [are]
   ↓
Neural Network Processing (Transformers)
   ↓
Probability Distribution: {you: 0.45, they: 0.25, we: 0.15, ...}
   ↓
Next Word Prediction: "you" (highest probability)
```

## Part 2: Tokenization - Breaking Text into Pieces

LLMs don't understand text like humans. They need to break it into smaller pieces called **tokens**.

### Why Tokenization?
- LLMs work with numbers, not letters
- Each token maps to a unique ID
- Helps the model understand word boundaries

### Example:
```
Text: "Hello, World!"

Character level:     H e l l o , [space] W o r l d !
Word level:          Hello , World !
Subword level:       Hell o , World !  ← Most LLMs use this

Token IDs (example): [7592, 13, 2160, 0]
```

### Why Subword Tokenization?
- Handles rare words better
- Example: "unbelievable" → ["un", "believe", "able"]
- Balances vocabulary size and word coverage

In [ ]:
# Install required libraries
!pip install transformers torch -q

In [ ]:
from transformers import AutoTokenizer
import json

# Load a tokenizer (we'll use DistilBERT - small and fast)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

print("=" * 60)
print("TOKENIZATION EXAMPLE")
print("=" * 60)

text = "Hello, how are you doing today?"
print(f"\nOriginal text: {text}")
print(f"Text length: {len(text)} characters")

# Tokenize the text
tokens = tokenizer.tokenize(text)
print(f"\nTokens: {tokens}")
print(f"Number of tokens: {len(tokens)}")

# Get token IDs
token_ids = tokenizer.encode(text, add_special_tokens=True)
print(f"\nToken IDs: {token_ids}")

# Decode back to text
decoded = tokenizer.decode(token_ids)
print(f"\nDecoded text: {decoded}")

print("\n" + "=" * 60)
print("TOKEN ↔️ ID MAPPING")
print("=" * 60)
for token, token_id in zip(tokens, token_ids[1:-1]):  # Skip [CLS] and [SEP]
    print(f"Token: '{token:15s}' → ID: {token_id}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

TOKENIZATION EXAMPLE

Original text: Hello, how are you doing today?
Text length: 31 characters

Tokens: ['hello', ',', 'how', 'are', 'you', 'doing', 'today', '?']
Number of tokens: 8

Token IDs: [101, 7592, 1010, 2129, 2024, 2017, 2725, 2651, 1029, 102]

Decoded text: [CLS] hello, how are you doing today? [SEP]

TOKEN ↔️ ID MAPPING
Token: 'hello          ' → ID: 7592
Token: ',              ' → ID: 1010
Token: 'how            ' → ID: 2129
Token: 'are            ' → ID: 2024
Token: 'you            ' → ID: 2017
Token: 'doing          ' → ID: 2725
Token: 'today          ' → ID: 2651
Token: '?              ' → ID: 1029


In [ ]:
print(type(tokens))

<class 'list'>


In [ ]:
# Let's see how different texts are tokenized
print("\n" + "=" * 60)
print("TOKENIZATION COMPARISON")
print("=" * 60)

examples = [
    "Hi",
    "Hello",
    "Unbelievable",
    "AI is awesome!",
    "123",
    "@user mentioned this"
]

for example in examples:
    tokens = tokenizer.tokenize(example)
    token_ids = tokenizer.encode(example)
    print(f"\nText: '{example}'")
    print(f"  Tokens: {tokens}")
    print(f"  Token count: {len(tokens)}")
    print(f"  Token IDs: {token_ids}")

## Part 3: Probability Calculation - How LLMs Predict

After processing tokens, an LLM outputs a **probability distribution** over all possible next words.

### The Process:
```
Input: "I love"
   ↓
LLM processes: [I] [love]
   ↓
Output scores for each word in vocabulary:
   cat:      2.5
   dog:      3.2
   you:      4.1  ← Highest
   pizza:    1.8
   ...
   ↓
Convert to probabilities using Softmax (0-1):
   cat:      0.08
   dog:      0.15
   you:      0.65  ← Highest probability!
   pizza:    0.05
   ...
```

### What is Softmax?
It converts raw scores (logits) into probabilities that sum to 1.

**Formula:**
```
probability(word_i) = exp(score_i) / sum(exp(all_scores))
```

**Simple example with 3 words:**
```
Scores:     [2.0, 1.0, 0.1]
After exp:  [7.39, 2.72, 1.10]
After sum:  11.21
Softmax:    [7.39/11.21, 2.72/11.21, 1.10/11.21]
            = [0.659, 0.243, 0.098]
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 60)
print("SOFTMAX FUNCTION EXPLAINED")
print("=" * 60)

def softmax(scores):
    """Convert scores to probabilities"""
    # Subtract max for numerical stability
    exp_scores = np.exp(scores - np.max(scores))
    return exp_scores / np.sum(exp_scores)

# Example scores from LLM
words = ['cat', 'dog', 'you', 'pizza', 'tea']
scores = np.array([2.5, 3.2, 4.1, 1.8, 2.0])
probabilities = softmax(scores)

print(f"\nContext: 'I love'")
print(f"\nWord → Score → Probability")
for word, score, prob in zip(words, scores, probabilities):
    print(f"{word:10s} → {score:5.1f} → {prob:6.1%}")

print(f"\nTotal probability: {np.sum(probabilities):.1%} ✓")
print(f"Most likely word: '{words[np.argmax(probabilities)]}' with {np.max(probabilities):.1%} probability")

In [ ]:
# Visualize the probability distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Bar chart of probabilities
axes[0].bar(words, probabilities, color='skyblue', edgecolor='navy')
axes[0].set_ylabel('Probability', fontsize=12)
axes[0].set_title('Next Word Probabilities\n(Context: "I love")', fontsize=13, fontweight='bold')
axes[0].set_ylim([0, max(probabilities) * 1.1])
for i, (word, prob) in enumerate(zip(words, probabilities)):
    axes[0].text(i, prob + 0.01, f'{prob:.1%}', ha='center', fontsize=10)

# Plot 2: How softmax converts scores to probabilities
axes[1].plot(scores, 'o-', label='Raw scores (logits)', linewidth=2, markersize=8)
axes[1].plot(probabilities * max(scores), 's-', label='Probabilities (scaled)', linewidth=2, markersize=8)
axes[1].set_ylabel('Value', fontsize=12)
axes[1].set_xlabel('Word Index', fontsize=12)
axes[1].set_title('Scores → Softmax → Probabilities', fontsize=13, fontweight='bold')
axes[1].set_xticks(range(len(words)))
axes[1].set_xticklabels(words, rotation=45)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nObservation: Softmax amplifies the highest score!")
print(f"Score difference: {max(scores) - min(scores):.1f}")
print(f"Probability difference: {max(probabilities) - min(probabilities):.1%}")

## Part 4: Using HuggingFace Models - Practical Examples

Now let's use real LLMs from HuggingFace! 🤗

### What is HuggingFace?
- **Hub**: A library of pre-trained models
- **Models**: BERT, GPT-2, DistilBERT, etc.
- **Tools**: Easy-to-use APIs for NLP tasks


### Example 1: Text-Genration:

In [ ]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

print("=" * 60)
print("TEXT GENERATION WITH GPT-2")
print("=" * 60)

# Load a text generation pipeline (GPT-2 is small and fast)
generator = pipeline('text-generation', model='gpt2')

# Generate text
prompt = "Give the information About Future technology?"
print(f"\nPrompt: '{prompt}'")
print("\nGenerating text...")

results = generator(prompt,
                    max_length=50,
                    num_return_sequences=3,
                    temperature=0.1)

for i, result in enumerate(results, 1):
    print(f"\n--- Generation {i} ---")
    print(result['generated_text'])

TEXT GENERATION WITH GPT-2


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt: 'Give the information About Future technology?'

Generating text...

--- Generation 1 ---
Give the information About Future technology?

The Future Technology Initiative (FTI) is a non-profit organization that provides funding for research and development of new technologies. The FTI is a non-profit organization that provides funding for research and development of new technologies. The FTI is a non-profit organization that provides funding for research and development of new technologies.

The FTI is a non-profit organization that provides funding for research and development of new technologies. The FTI is a non-profit organization that provides funding for research and development of new technologies.

The FTI is a non-profit organization that provides funding for research and development of new technologies.

The FTI is a non-profit organization that provides funding for research and development of new technologies.

The FTI is a non-profit organization that provides fundi

In [ ]:
# Let's see the step-by-step process
print("\n" + "=" * 60)
print("STEP-BY-STEP TEXT GENERATION")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

prompt = "The future of AI"
tokens = tokenizer.encode(prompt)

print(f"\nPrompt: '{prompt}'")
print(f"Tokens: {tokens}")
print(f"Decoded tokens: {[tokenizer.decode([t]) for t in tokens]}")

print(f"\n{'Step':<6} {'Predicted Token':<20} {'Probability':<15}")
print("-" * 45)

# Generate one word at a time
import torch

with torch.no_grad():
    input_ids = torch.tensor([tokens])

    for step in range(5):  # Generate 5 words
        outputs = model(input_ids)
        logits = outputs.logits[0, -1, :]  # Last token's logits

        # Apply softmax to get probabilities
        probs = torch.softmax(logits, dim=-1)
        next_token_id = torch.argmax(probs).item()
        next_token_prob = probs[next_token_id].item()
        next_token_text = tokenizer.decode([next_token_id])

        print(f"{step+1:<6} {next_token_text:<20} {next_token_prob:<15.1%}")

        # Append to input for next iteration
        input_ids = torch.cat([input_ids, torch.tensor([[next_token_id]])], dim=1)

print(f"\nFinal text: {tokenizer.decode(input_ids[0])}")

### Example 2: Masked Language Modeling (Fill in the blanks)

In [ ]:
print("\n" + "=" * 60)
print("MASKED LANGUAGE MODELING (Fill the Blanks)")
print("=" * 60)

# Load masked LM pipeline
mask_pipeline = pipeline('fill-mask', model='distilbert-base-uncased')

# [MASK] is a special token that the model tries to fill
texts = [
    "Python is a [MASK] programming language.",
    "The capital of France is [MASK].",
    "Transformers use [MASK] attention mechanisms.",
    "I [MASK] machine learning very much."
]

print()
for text in texts:
    print(f"Input: {text}")
    results = mask_pipeline(text, top_k=3)  # Get top 3 predictions

    for i, result in enumerate(results, 1):
        filled = result['sequence']
        score = result['score']
        token = result['token_str']
        print(f"  {i}. Prediction: '{token}' (Confidence: {score:.1%})")
        print(f"     Filled: {filled}")
    print()

## Part 6: Key Concepts Summary

Let's recap everything we learned! 📚

In [ ]:
print("\n" + "=" * 70)
print("🎯 KEY CONCEPTS SUMMARY")
print("=" * 70)

summary = {
    "1. TOKENIZATION": {
        "What": "Breaking text into smaller pieces (tokens)",
        "Why": "LLMs work with numbers, not letters",
        "Example": "'Hello, world!' → ['hello', ',', 'world', '!']",
        "Types": "Character-level, Word-level, Subword-level",
        "Best for LLMs": "Subword tokenization (BPE, SentencePiece)"
    },

    "2. LLM ARCHITECTURE": {
        "Core": "Transformer with attention mechanisms",
        "Process": "Tokens → Embeddings → Neural layers → Output",
        "Key idea": "Attention allows model to focus on relevant parts",
        "Training": "Predicts next word from previous context",
        "Popular models": "GPT, BERT, T5, LLaMA, Mistral"
    },

    "3. PROBABILITY CALCULATION": {
        "Raw scores": "Logits from neural network",
        "Convert to probabilities": "Use Softmax function",
        "Softmax formula": "P(i) = exp(score_i) / sum(exp(all_scores))",
        "Result": "Distribution over vocabulary (sums to 1)",
        "Interpretation": "Higher probability = more likely next word"
    },

    "4. USING HUGGINGFACE": {
        "Installation": "pip install transformers torch",
        "Basic usage": "pipeline('task', model='model_name')",
        "Common tasks": "Generation, Classification, QA, Summarization",
        "Models": "Pre-trained on large datasets, ready to use!",
        "Customization": "Can fine-tune on your own data"
    }
}

for concept, details in summary.items():
    print(f"\n{concept}")
    print("-" * 70)
    for key, value in details.items():
        print(f"  • {key}: {value}")

print("\n" + "=" * 70)

In [ ]:
""" I am a student studying in college"""-->> Casual Language model (or) Base models.